In [1]:
from pathlib import Path
from typing import Self

import zarr
import numpy as np
import spotpy

In [2]:
def build_parameter_list(setup, parameter_dim: int | None = None, parameter_dim_name: str | None = None):
    cls = type(setup)
    class_vars = vars(cls).items()
    parameters = []
    for attrname, attrobj in class_vars:
        # Check if it is a spotpy parameter
        if isinstance(attrobj, spotpy.parameter.Base):
            if not attrobj.name:
                attrobj.name = attrname

            if attrobj.rndfunctype != "MultiDimUniform":
                if (parameter_dim is None) or (parameter_dim_name is None):
                    raise ValueError("No parameter dimension or parameter dimension name provided")
                attrobj.shape = parameter_dim
                attrobj.dim_names = parameter_dim_name

            if attrobj.rndfunctype == 'List':
                if isinstance(attrobj.values, list):
                    attrobj.values = np.array(attrobj.values)
                if len(attrobj.values.shape) == 1:
                    if attrobj.values.shape[0] != parameter_dim:
                        raise ValueError(
                            "The provided parameter dimensions do not match the length of the List Parameter values.")
                    attrobj.maxbound = np.nanmax(attrobj.values)
                    attrobj.minbound = np.nanmin(attrobj.values)
                elif len(attrobj.values.shape) == 2:
                    if attrobj.values.shape[1] != parameter_dim:
                        raise ValueError(
                            "The provided parameter dimensions do not match the length of the List Parameter values.")
                    attrobj.maxbound = np.nanmax(attrobj.values, axis=0)
                    attrobj.minbound = np.nanmin(attrobj.values, axis=0)
                else:
                    raise NotImplementedError("List Parameter type with > 2 dimensions is not supported.")

            # Add parameter
            parameters.append(attrobj)

    return parameters

In [3]:
def find_chunks(dims: tuple[int], chunks: tuple[int | None] | None = None, max_size: int | float = 10.0, dtype: str = 'float64'):
    """Find multidimensional chunks for an array size threshold"""
    nelems = np.prod(dims)
    ndact = len(dims)
    if chunks is not None:
        if len(dims) != len(chunks):
            raise ValueError("Length of dimensions and chunks is unequal")
        achnk = []
        for i, c in enumerate(chunks):
            if c is None:
                achnk.append(dims[i])
            elif c > dims[i]:
                achnk.append(dims[i])
                ndact -= 1
            else:
                achnk.append(c)
                ndact -= 1
        achnk = tuple(achnk)
    else:
        achnk = dims

    nelems = np.prod(achnk)

    if (dtype == 'float64') or (dtype == 'int64'):
        x = 8
    elif (dtype == 'float32') or (dtype == 'int32'):
        x = 4
    elif dtype[:-1] == '<U':
        x = int(dtype[-1]) * 4
    elif dtype[:-1] == '|S':
        x = int(dtype[-1])
    else:
        x = 255
    
    nbytes = nelems * x
    tbytes = max_size * 1e6
    if nbytes < tbytes:
        tbytes = nbytes
    kn = tbytes / nbytes
    k = kn**(1 / ndact)

    fchnk = []
    if chunks is not None:
        for i, d in enumerate(chunks):
            if d is not None:
                fchnk.append(d)
            else:
                fchnk.append(int(dims[i] * k))
    else:
        for d in dims:
            fchnk.append(int(d * k))

    return tuple(fchnk)

In [62]:
ch = find_chunks((1000, 1800, 45000), chunks=(1000, 1, None), max_size=50, dtype='float32')

In [63]:
ch

(1000, 1, 12500)

In [64]:
np.prod((1000, 1800, 45000)) / np.prod(ch)

np.float64(6480.0)

In [61]:
np.prod(ch)

np.int64(10000000)

In [26]:
class ZarrDb:

    def __init__(self, 
                 dbpath: str | Path | None = None,
                 dbname: str = 'zarr_db',
                 overwrite: bool = False,
                 **kwargs):

        if dbpath is not None:
            self.dbpath = Path(dbpath)
            if (self.dbpath.exists()) & (not overwrite):
                try:
                    z = zarr.open(dbpath)
                    self.zstore = z.store
                    self.dbname = z.attrs['dbname']
                    self.dims = z.attrs['dims']
                except Exception as e:
                    print(f"An error occurred opening the database: {e}")
            else:
                if self.dbpath.suffix == '.zip':
                    #self.zstore = zarr.storage.ZipStore(self.dbpath, mode='w', **kwargs)
                    #self.dbname = dbname
                    #zarr.create_group(store=self.zstore)
                    #with zarr.open_group(self.zstore, mode='w') as g:
                        #g.attrs.update({'dbname': self.dbname, 'dims': {}})
                    raise NotImplementedError("The zarr ZipStore has not been implemented yet, to get a zipped file, create a LocalStore and zip when database is complete.")
                else:
                    self.zstore = zarr.storage.LocalStore(self.dbpath, read_only=False, **kwargs)
                    self.dbname = dbname
                    zarr.create_group(store=self.zstore, overwrite=True)
                    g = zarr.open_group(self.zstore, mode='r+')
                    g.attrs.update({'dbname': self.dbname, 'dims': {}})
        else:
            self.dbpath = dbpath
            self.zstore = zarr.storage.MemoryStore(**kwargs)
            self.dbname = dbname
            zarr.create_group(store=self.zstore, overwrite=True)
            g = zarr.open_group(self.zstore, mode='r+')
            g.attrs.update({'dbname': self.dbname, 'dims': {}})

    @property
    def root(self) -> zarr.Group:
        """The root group, or top level, of the zarr store opened as read-only
        """
        return zarr.open_group(self.zstore, mode='r')

    @property
    def dimensions(self) -> dict:
        """A dictionary of dimension names and sizes found in the database
        """
        return self.root.attrs['dims']

    def create_array(self, 
                     path: str,
                     dimension_names: list,
                     shape: tuple | None = None, 
                     data: np.ndarray | None = None,
                     chunks: tuple | str = 'auto',
                     data_type: str = 'float32',
                     **kwargs):
        """Wrapper around zarr.create_array() method

        Creates a new array at the specified path within the zarr database. Any additional key word arguments
        accepted by zarr.create_array() can be included. Either data or shape must be provided. If no data is
        provided, the empty array will be created. The default behavior is to assume a floating point array, if
        another data type is required, change data_type and fill_value.

        Args:
            path:
            dimension_names:
            shape:
            data:
            chunks:
            data_type:
        """
        if (shape is None) & (data is None):
            raise ValueError("Data array or a shape tuple required, none provided.")

        if shape is not None:
            a_shp = shape
        else:
            a_shp = data.shape

        if len(dimension_names) != len(a_shp):
            raise ValueError(f"Length of dimension names {len(dimension_names)} does not match number of dimensions {len(a_shp)}")

        if 'overwrite' not in kwargs:
            kwargs['overwrite'] = True

        if 'fill_value' not in kwargs:
            kwargs['fill_value'] = np.nan

        if data is not None:
            data_type = None

        zarr.create_array(store=self.zstore, name=path, shape=shape, data=data, chunks=chunks, dtype=data_type, dimension_names=dimension_names, **kwargs)
        root = zarr.open_group(self.zstore, mode='r+')
        rdims = self.root.attrs['dims']
        for c in zip(dimension_names, a_shp):
            if c[0] in rdims:
                continue
            else:
                rdims.update(dict([c]))

        root.attrs['dims'] = rdims

    def delete_array(self, path: str):
        # needs to loop through root members and determine unique dimensions of all arrays
        #   after a deletion occurs
        return None

    def insert(self, path: str, data: np.ndarray, idx: list[int | np.ndarray | list | slice | None] | np.ndarray | None = None):
        """Insert data into an array in the database 
        """
        root = zarr.open_group(store=self.zstore, mode='r+')
        if idx is not None:
            if isinstance(idx, np.ndarray):
                idx = idx.tolist()
            indx_vals = []
            for i, ix in enumerate(idx):
                if isinstance(ix, int):
                    a = [ix]
                elif (ix == slice(None)) | (ix is None):
                    a = np.arange(self.root[path].shape[i]).tolist()
                else:
                    a = ix
                indx_vals.append(a)
            sel = np.ix_(*indx_vals)
            root[path][sel] = data
        else:
            root[path][:] = data
    
    def to_xarray(self, group: str | None = None):
        return None

    def __str__(self) -> str:
        """Returns a string representation of the database tree
        """

        return str(self.root.tree())


class MscuaDb(ZarrDb):

    def __init__(self, *args, **kwargs):
        super(MscuaDb, self).__init__(*args, **kwargs)
        # initialize the zarr groups and arrays here?
        self.parameters = None
        self.active_iteration = None

    @property
    def parameter_names(self) -> dict:
        pdct = {}
        if len(list(self.root.groups())) == 0:
            return pdct
        else:
            for g in self.root.groups():
                try:
                    pdct[g[0]] = g[1].attrs['parameters']
                except KeyError:
                    continue
            
            return pdct

    @property
    def target_datasets(self) -> dict:
        tddct = {}
        if len(list(self.root.groups())) == 0:
            return tddct
        else:
            for g in self.root.groups():
                try:
                    tddct[g[0]] = g[1].attrs['target_datasets']
                except KeyError:
                    continue
            
            return tddct

    @property
    def objective_functions(self) -> dict:
        obdct = {}
        if len(list(self.root.groups())) == 0:
            return obdct
        else:
            for g in self.root.groups():
                try:
                    obdct[g[0]] = g[1].attrs['objective_functions']
                except KeyError:
                    continue
            
            return obdct

    @property
    def calibration_dimension(self) -> dict:
        caldimdct = {}
        if len(list(self.root.groups())) == 0:
            return caldimdct
        else:
            for g in self.root.groups():
                try:
                    caldimdct[g[0]] = g[1].attrs['calibration_dimension']
                except KeyError:
                    continue
            
            return caldimdct
        
    def add_iteration(self, 
                      params: list[spotpy.parameter.Base],
                      cal_targets: dict,
                      caldim_name: str,
                      nobs: dict, 
                      reps: int = 1000, 
                      iter_name: str | None = None):
        
        self.parameters = params
        if iter_name is None:
            n_iters = len([i for i in self.root.group_keys()])
            iter_name = f"Iteration{n_iters + 1}"
        self.active_iteration = iter_name

        zarr.create_group(store=self.zstore, path=iter_name, overwrite=True)
        # create attributes dictionary for top level of iteration
        iter_attrs = {'parameters': [p.name for p in self.parameters],
                      'target_datasets': list(cal_targets.keys()),
                      'objective_functions': {},
                      'calibration_dimension': caldim_name}
        for dk, dv in cal_targets.items():
            iter_attrs['objective_functions'][dk] = list(dv.keys())
        # create static threshold scalars
        self.create_array(path=f"{iter_name}/Thresholds/pfactor", 
                                                            dimension_names=['scalar'], 
                                                            shape=(1,), 
                                                            chunks=(1,))
        self.create_array(path=f"{iter_name}/Thresholds/min_refined_params", 
                                                                    dimension_names=['scalar'], 
                                                                    shape=(1,), 
                                                                    chunks=(1,))
        # Create parameter dimensioned arrays
        for param in self.parameters:
            if caldim_name not in param.dim_names:
                raise ValueError(f"DatabaseError: {param.name} parameter is missing the {caldim_name} dimension")     
            self.create_array(path=f"{iter_name}/Parameters/InitialSamples/{param.name}", 
                                dimension_names=['repetitions'] + list(param.dim_names), 
                                shape=(reps,) + param.shape, 
                                chunks=find_chunks(dims=(reps,)+param.shape, chunks=(1, None), dtype='float32'))
            self.create_array(path=f"{iter_name}/Parameters/RefinedSets/{param.name}", 
                                            dimension_names=['repetitions'] + list(param.dim_names), 
                                            shape=(reps,) + param.shape, 
                                            chunks=find_chunks(dims=(reps,)+param.shape, chunks=(1, None), dtype='float32'))
            self.create_array(path=f"{iter_name}/Parameters/RefinedSets/Best/{param.name}", 
                                                                dimension_names=list(param.dim_names), 
                                                                shape=param.shape, 
                                                                chunks=param.shape)
        # Create best parameter arrays (by target dataset and obj function), objective function arrays, best obj function arrays, and obj function thresholds
        for kt, vt in cal_targets.items():
            for kb, vb in vt.items():
                for param in self.parameters:
                    self.create_array(path=f"{iter_name}/Parameters/InitialSamples/Best/{kt}/{kb}/{param.name}", 
                                                dimension_names=list(param.dim_names), 
                                                shape=param.shape, 
                                                chunks=param.shape)
                self.create_array(path=f"{iter_name}/ObjectiveFunctions/{kt}/{kb}", 
                                              dimension_names=['repetitions', caldim_name], 
                                              shape=(reps, self.dimensions[caldim_name]),
                                              chunks=find_chunks(dims=(reps, self.dimensions[caldim_name]), chunks=(1, None), dtype='float32'))
                self.create_array(path=f"{iter_name}/ObjectiveFunctions/{kt}/Best/{kb}", 
                                                              dimension_names=[caldim_name], 
                                                              shape=(self.dimensions[caldim_name],),
                                                              chunks=(self.dimensions[caldim_name],))
                self.create_array(path=f"{iter_name}/Thresholds/ObjectiveFunctions/{kt}/{kb}", 
                                    dimension_names=['scalar'], 
                                    data=np.array([cal_targets[kt][kb]], dtype=np.float32), 
                                    chunks=(1,))
                self.create_array(path=f"{iter_name}/Simulations/Best/{kt}/{kb}", 
                                    dimension_names=['observations', caldim_name], 
                                    shape=(nobs[kt], self.dimensions[caldim_name]),
                                    chunks=find_chunks(dims=(nobs[kt], self.dimensions[caldim_name]), chunks=(nobs[kt], None), dtype='float32'))
            self.create_array(path=f"{iter_name}/Simulations/Best/{kt}_best", 
                                dimension_names=['observations', caldim_name], 
                                shape=(nobs[kt], self.dimensions[caldim_name]),
                                chunks=find_chunks(dims=(nobs[kt], self.dimensions[caldim_name]), chunks=(nobs[kt], None), dtype='float32'))
            self.create_array(path=f"{iter_name}/Metrics/{kt}/pfactor", 
                                dimension_names=[caldim_name], 
                                shape=(self.dimensions[caldim_name],),
                                chunks=(self.dimensions[caldim_name],))
            self.create_array(path=f"{iter_name}/Metrics/{kt}/rfactor", 
                                dimension_names=[caldim_name], 
                                shape=(self.dimensions[caldim_name],),
                                chunks=(self.dimensions[caldim_name],))
            self.create_array(path=f"{iter_name}/Metrics/{kt}/95ppu_upper", 
                                dimension_names=['observations', caldim_name], 
                                shape=(nobs[kt], self.dimensions[caldim_name]),
                                chunks=find_chunks(dims=(nobs[kt], self.dimensions[caldim_name]), chunks=(nobs[kt], None), dtype='float32'))
            self.create_array(path=f"{iter_name}/Metrics/{kt}/95ppu_lower", 
                                dimension_names=['observations', caldim_name], 
                                shape=(nobs[kt], self.dimensions[caldim_name]),
                                chunks=find_chunks(dims=(nobs[kt], self.dimensions[caldim_name]), chunks=(nobs[kt], None), dtype='float32'))

        root = zarr.open_group(store=self.zstore, mode='r+')
        root[iter_name].attrs.update(iter_attrs)
    
    def add_dimension_values(self, dim_data: dict):

        for k, v in dim_data.items():
            if k not in self.dimensions:
                raise KeyError(f"{k} not found in database dimensions")
            self.create_array(path=f"{k}",
                              dimension_names=[k],
                              data=v,
                              chunks=v.shape,
                              fill_value=None)
    
    def get_simulation_results(self, 
                               sim_output_name: str, 
                               iteration: str | None = None, 
                               rep: int | list | None = None, 
                               obs_id: int | list | None = None, 
                               caldim_id: int | list | None = None):
        if iteration is None:
            if self.active_iteration is None:
                print("No active iteration")
                return None
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            print(f"Iteration: '{iteration}' not found in database")
            return None

        sgroup = self.root[f"{iteration}/Simulations"]
        if sim_output_name not in sgroup:
            print("Simulation data not saved in database")
            return None        

        rarray = self.root[f"{iteration}/Simulations/{sim_output_name}"]
        if rep is None:
            rep = np.arange(rarray.shape[0])
        if isinstance(rep, int):
            rep = [rep]
        if obs_id is None:
            obs_id = np.arange(rarray.shape[1])
        if isinstance(obs_id, int):
            obs_id = [obs_id]
        if caldim_id is None:
            caldim_id = np.arange(rarray.shape[2])
        if isinstance(caldim_id, int):
            caldim_id = [caldim_id]
        indx_vals = [rep, obs_id, caldim_id]
        sel = np.ix_(*indx_vals)
        
        return rarray[sel]
    
    def get_parameter_samples(self,
                              param_name: str,
                              iteration: str | None = None, 
                              rep: int | list | None = None, 
                              caldim_id: int | list | None = None):

        if iteration is None:
            if self.active_iteration is None:
                print("No active iteration")
                return None
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            print(f"Iteration: '{iteration}' not found in database")
            return None

        rarray = self.root[f"{iteration}/Parameters/InitialSamples/{param_name}"]
        if rep is None:
            rep = np.arange(rarray.shape[0])
        if isinstance(rep, int):
            rep = [rep]
        if caldim_id is None:
            caldim_id = np.arange(rarray.shape[rarray.metadata.dimension_names.index(self.calibration_dimension[iteration])])
        if isinstance(caldim_id, int):
            caldim_id = [caldim_id]
        indx_vals = [rep, caldim_id]
        sel = np.ix_(*indx_vals)

        return rarray[sel]

    def get_objective_func_values(self, 
                                  obj_func_name: str, 
                                  target_dset: str,
                                  iteration: str | None = None,
                                  rep: int | list | None = None, 
                                  caldim_id: int | list | None = None):

        if iteration is None:
            if self.active_iteration is None:
                print("No active iteration")
                return None
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            print(f"Iteration: '{iteration}' not found in database")
            return None

        rarray = self.root[f"{iteration}/ObjectiveFunctions/{target_dset}/{obj_func_name}"]
        if rep is None:
            rep = np.arange(rarray.shape[0])
        if isinstance(rep, int):
            rep = [rep]
        if caldim_id is None:
            caldim_id = np.arange(rarray.shape[rarray.metadata.dimension_names.index(self.calibration_dimension[iteration])])
        if isinstance(caldim_id, int):
            caldim_id = [caldim_id]
        indx_vals = [rep, caldim_id]
        sel = np.ix_(*indx_vals)

        return rarray[sel]
    
    def get_refined_parameters(self, 
                               param_name: str,
                               iteration: str | None = None, 
                               rep: int | list | None = None, 
                               caldim_id: int | list | None = None):
        if iteration is None:
            if self.active_iteration is None:
                print("No active iteration")
                return None
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            print(f"Iteration: '{iteration}' not found in database")
            return None

        rarray = self.root[f"{iteration}/Parameters/InitialSamples/{param_name}"]
        if rep is None:
            rep = np.arange(rarray.shape[0])
        if isinstance(rep, int):
            rep = [rep]
        if caldim_id is None:
            caldim_id = np.arange(rarray.shape[rarray.metadata.dimension_names.index(self.calibration_dimension[iteration])])
        if isinstance(caldim_id, int):
            caldim_id = [caldim_id]
        indx_vals = [rep, caldim_id]
        sel = np.ix_(*indx_vals)

        return rarray[sel]

    def save_parameter(self,
                       param_name: str,
                       param_type: str,
                       param_data: np.ndarray,
                       iteration: str | None = None,
                       rep: int | list | None = None, 
                       caldim_id: int | list | None = None):

        if iteration is None:
            if self.active_iteration is None:
                raise ValueError("No iteration specified to save to")
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            raise KeyError(f"Iteration: '{iteration}' not found in database")

        if param_type == 'initial':
            self.insert(path=f"{iteration}/Parameters/InitialSamples/{param_name}", data=param_data, idx=[rep, caldim_id])
        elif param_type == 'refined':
            self.insert(path=f"{iteration}/Parameters/RefinedSets/{param_name}", data=param_data, idx=[rep, caldim_id])

    def save_objective_func(self, 
                            obj_func_name: str, 
                            target_dset: str,
                            objfunc_data: np.ndarray, 
                            iteration: str | None = None,
                            rep: int | list | None = None, 
                            caldim_id: int | list | None = None):

        if iteration is None:
            if self.active_iteration is None:
                raise ValueError("No iteration specified to save to")
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            raise KeyError(f"Iteration: '{iteration}' not found in database")

        self.insert(path=f"{iteration}/ObjectiveFunctions/{target_dset}/{obj_func_name}", data=objfunc_data, idx=[rep, caldim_id])

    def save_simulation(self, 
                        sim_output_name: str, 
                        sim_data: np.ndarray, 
                        iteration: str | None = None,
                        rep: int | list | None = None, 
                        row_id: int | list | None = None, 
                        caldim_id: int | list | None = None):

        if iteration is None:
            if self.active_iteration is None:
                raise ValueError("No iteration specified to save to")
            else:
                iteration = self.active_iteration

        if iteration not in list(self.root.group_keys()):
            raise KeyError(f"Iteration: '{iteration}' not found in database")

        self.insert(path=f"{iteration}/Simulations/{sim_output_name}", data=sim_data, idx=[rep, row_id, caldim_id])

    def get_xarray(self, iteration: str | None = None, group: str | None = None):
        return self.to_xarray()

In [27]:
test = Path('D:/Projects/Test')
db = MscuaDb(dbname='Test', dbpath=None)

In [28]:
plst = [spotpy.parameter.Uniform(name='exp', low=0.0, high=8.5), spotpy.parameter.Uniform(name='coef', low=0.03, high=1.2)]
nplst = []
for p in plst:
    p.shape = (300,)
    p.dim_names = ('curve',)
    nplst.append(p)
db.add_iteration(params=nplst,
                 cal_targets={'flow': {'nse': 0.5}, 'temp': {'rmse': 2.5, 'nrmse': 0.1}},
                 caldim_name='curve',
                 nobs={'flow': 300, 'temp': 300},
                 reps=500)

In [29]:
db.add_dimension_values({'curve': np.arange(db.dimensions['curve'], dtype=np.int32)})

In [32]:
db.save_parameter('coef', 'initial', np.random.random((500, 300)) * 2)

In [36]:
db.get_parameter_samples('exp')

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(500, 300), dtype=float32)

In [30]:
db.get_objective_func_values('rmse', 'temp', rep=0)

array([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan, nan

In [65]:
db.insert(path='Iteration2/Parameters/InitialSamples/coef', data=np.ones(db.dimensions['curve']), idx=[0, slice(None)])

In [67]:
db.root['Iteration2/Parameters/InitialSamples/coef'][:]

array([[ 1.,  1.,  1., ...,  1.,  1.,  1.],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(500, 300), dtype=float32)

In [59]:
print(db)

/
├── Iteration1
│   ├── ObjectiveFunctions
│   │   └── flow
│   │       ├── Best
│   │       │   └── nse (300,) float32
│   │       └── nse (500, 300) float32
│   ├── Parameters
│   │   ├── InitialSamples
│   │   │   ├── Best
│   │   │   │   └── flow
│   │   │   │       └── nse
│   │   │   │           ├── coef (300,) float32
│   │   │   │           └── exp (300,) float32
│   │   │   ├── coef (500, 300) float32
│   │   │   └── exp (500, 300) float32
│   │   └── RefinedSets
│   │       ├── Best
│   │       │   ├── coef (300,) float32
│   │       │   └── exp (300,) float32
│   │       ├── coef (500, 300) float32
│   │       └── exp (500, 300) float32
│   └── Thresholds
│       ├── ObjectiveFunctions
│       │   └── flow
│       │       └── nse (1,) float32
│       ├── min_refined_params (1,) float32
│       └── pfactor (1,) float32
└── Iteration2
    ├── Metrics
    │   ├── flow
    │   │   ├── 95ppu_lower (300, 300) float32
    │   │   ├── 95ppu_upper (300, 300) float32
    │   │   ├── 

In [19]:
g = zarr.create_group({})
g.create_array('Iteration1/Parameters/coef', data=np.ones((10, 10)), chunks=(10, 10), dimension_names=['reps', 'curves'])

<Array memory://1821420826688/Iteration1/Parameters/coef shape=(10, 10) dtype=float64>

In [ ]:
pgroup = g['Iteration1/Parameters']

In [20]:
ar = g['Iteration1/Parameters/coef']

In [23]:
ar.shape

(10, 10)

In [22]:
ar.metadata.dimension_names.index('reps')

0

In [18]:
'exp' in pgroup

False

In [158]:
class Anumber:

    def __init__(self, thenum, itname):
        self.number = thenum
        self.num_name = itname


class TestThing:
    aparam = Anumber(10, 'LabelNotToChange')

    def __init__(self):
        print("it is done")

def afunc(x, y):
    return {afunc.__name__: x * y}

In [2]:
a = np.array([[1,2,3,3], [1, np.nan, 3, 5], [np.nan, np.nan, np.nan, np.nan], [4, 3, 9, np.nan]])

In [21]:
'a' in ('b', 'c', 'd')

False

In [18]:
np.ones((5,5)) * 10

array([[10., 10., 10., 10., 10.],
       [10., 10., 10., 10., 10.],
       [10., 10., 10., 10., 10.],
       [10., 10., 10., 10., 10.],
       [10., 10., 10., 10., 10.]])

In [19]:
np.random.random()

0.5589436668164136

In [11]:
a[np.nanargmax(a, axis=0), np.arange(a.shape[1])]

array([4., 3., 9., 5.])

In [195]:
print(a.aparam.number)
print(a.aparam.num_name)

10
LabelNotToChange


In [196]:
a.aparam.__init__(42, a.aparam.num_name)

In [197]:
print(a.aparam.number)
print(a.aparam.num_name)

42
LabelNotToChange


In [164]:
db.create_array(path='Parameters/coef', dimension_names=['reps', 'features'], shape=(1000, 300), chunks=(1, 300))

In [165]:
db.insert(path='Parameters/coef', data=np.ones(300), idx=[1, None])

In [42]:
g = zarr.create_group({})
#g.create_group('Iteration1')
#g.create_group('Iteration2')

In [43]:
for i in g.groups():
    try:
        t = i[1].attrs['parameters']
    except KeyError:
        t = {}

In [45]:
len(list(g.groups()))

0

In [208]:
for i in db.root.members(max_depth=None):
    if type(i[1]) != zarr.Group:
        continue
    else:
        print(i[0])

Parameters


In [96]:
dn = ['c', 'b']
shp = (50, 200)

In [ ]:
d = {}

In [97]:
for c in zip(dn, shp):
    if c[0] in d:
        continue
    else:
        d.update(dict([c]))

In [144]:
g = zarr.open_group(store=db.zstore, mode='r+')

In [147]:
g.attrs['dims'] = {}

In [148]:
g.attrs['dims']

{}

In [31]:
db.create_array('Parameters/coef', shape=(db.dims['reps'], db.dims['features']), chunks=(1, db.dims['features']))

In [32]:
g = zarr.create_group(store={})
a = g.create_array('tarray', shape=(1000, 300), chunks=(1, 300), dtype='float32')

In [36]:
g['tarray'].shape

(1000, 300)

In [38]:
np.ix_([0,1], [1,2,3])

(array([[0],
        [1]]),
 array([[1, 2, 3]]))

In [39]:
db.root['Parameters/coef'][np.ix_([0,1], [1,2,3])]

array([[nan, nan, nan],
       [nan, nan, nan]], dtype=float32)

In [23]:
t[(1,0)]

np.int64(2)

In [67]:
z = zarr.storage.MemoryStore({})

In [70]:
zarr.create_group(store=z)

<Group memory://2512586408128>

In [73]:
g = zarr.open_group(store=z)

In [74]:
g.tree()

/

In [7]:
zarr.create_group(store=db.zstore, path='Iteration_1')

<Group file://D:/Projects/Test/Iteration_1>

In [12]:
g = zarr.open_group(db.zstore)

In [25]:
zarr.create_array(store=db.zstore, name='/Parameters/coef', shape=(db.dims['reps'], db.dims['features']), chunks=(1, db.dims['features']), overwrite=True, dtype='float32')

<Array memory://1867193642240/Parameters/coef shape=(1000, 300) dtype=float32>

In [26]:
print(str(g.tree()))

/
└── Parameters
    └── coef (1000, 300) float32



In [122]:
t = zarr.create_group(store=test)

In [129]:
s = zarr.storage.ZipStore(test, mode='w')

In [130]:
zarr.create_group(s, path='Group')

<Group zip://D:\Projects\Test.zip/Group>

In [106]:
t.create_array('anarray', data=np.ones(10))

<Array memory://2347463588672/anarray shape=(10,) dtype=float64>

In [107]:
t.tree()

/
└── anarray (10,) float64

In [108]:
w = zarr.storage.WrapperStore(t.store)

In [113]:
t.store.get('anarray')

<coroutine object MemoryStore.get at 0x0000022290A29C40>

In [111]:
w.is_empty()

TypeError: WrapperStore.is_empty() missing 1 required positional argument: 'prefix'

In [76]:
tpar = MdUniform(name='tpar', low=0, high=10, shape=(12, 100), dim_names=('nmonths', 'features'), doc='A test parameter')